# Audit Detector, Size, and Inference Sensitivity

Connects detector sensitivity to size-specific ORDEM responses and tests contamination and inference-calibration assumptions.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import hashlib
import json
import math
import os
import tempfile
import time
import warnings
import zipfile

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter1d
from scipy.optimize import minimize
from scipy.special import gammaln
from scipy.stats import chisquare, norm

try:
    from IPython.display import display
except ImportError:
    def display(value):
        print(value)


RAW_DIR = PROJECT_ROOT / 'ordem_raw'
REFINEMENT_ZIP = RAW_DIR / 'job-b9824543-c020-42b7-b747-61b298670ce4.zip'
OLD_ACTION_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / '33aa70ca_80_plus_baseline'
OLD_RESPONSE_PATH = OLD_ACTION_DIR / 'ordem_action_response_matrix.csv.gz'
FINE_GRID_DIR = PROJECT_ROOT / 'phase1_outputs' / 'ordem_action_library' / 'b982_fine_elevation_49'
FINE_RESPONSE_PATH = FINE_GRID_DIR / 'ordem_fine_elevation_response_matrix_49.csv'
EVENT_PATH = PROJECT_ROOT / 'phase0_outputs' / 'catalog_matching' / 'ca_cfar_1p75_24h_final_audited' / 'final_consolidated_catalog_association.csv'
DETECTOR_DIR = PROJECT_ROOT / 'phase0_outputs' / 'detector_benchmark_raw_swerling_v2'
SELECTED_DETECTOR_PATH = DETECTOR_DIR / 'SELECTED_DETECTOR.json'
DETECTOR_FOLDS_PATH = DETECTOR_DIR / 'six_fold_held_out_evaluation.csv'
DETECTOR_RANKING_PATH = DETECTOR_DIR / 'detector_operating_point_ranking.csv'
PUBLICATION_DIR = PROJECT_ROOT / 'phase3_outputs' / 'publication_lock_validation' / 'robust_lock_598bd992a783'
CATALOG_RECOVERY_PATH = PUBLICATION_DIR / 'catalog_recovery_by_rcs.csv'
CATALOG_SUMMARY_PATH = PUBLICATION_DIR / 'catalog_validation_summary.json'
PREVIOUS_CONFIG_CANDIDATES = sorted(
    (PROJECT_ROOT / 'phase3_outputs' / 'final_bayesian_policy').glob('fine_grid_*/final_experiment_config.json')
)

EXPECTED_REFINEMENT_SHA256 = 'eade76d0fcca361e662b8a5da48b67baa809530d3e028b825da445752e75a10a'
RANGE_MIN_KM = 1026.639
RANGE_MAX_KM = 1084.050
ALTITUDE_EDGES = np.arange(915.0, 1065.0 + 0.1, 15.0)
ALTITUDE_CENTERS = 0.5 * (ALTITUDE_EDGES[:-1] + ALTITUDE_EDGES[1:])
ELEVATIONS = np.arange(65.0, 89.0 + 0.01, 0.5)
N_SHELLS = len(ALTITUDE_CENTERS)
N_ACTIONS = len(ELEVATIONS)
CENTER_ACTION = int(np.where(np.isclose(ELEVATIONS, 75.0))[0][0])
BUDGET = int(os.environ.get('DEBRIS_CLOSURE_BUDGET', '24'))
N_PRIMARY = int(os.environ.get('DEBRIS_CLOSURE_PRIMARY_REPS', '200'))
N_SENSITIVITY = int(os.environ.get('DEBRIS_CLOSURE_SENS_REPS', '100'))
N_SBC = int(os.environ.get('DEBRIS_CLOSURE_SBC_WORLDS', '100'))
SBC_BURN = int(os.environ.get('DEBRIS_CLOSURE_SBC_BURN', '500'))
SBC_DRAWS = int(os.environ.get('DEBRIS_CLOSURE_SBC_DRAWS', '1000'))
SBC_THIN = int(os.environ.get('DEBRIS_CLOSURE_SBC_THIN', '2'))
N_RHAT_WORLDS = int(os.environ.get('DEBRIS_CLOSURE_RHAT_WORLDS', '8'))
BOOTSTRAP_DRAWS = int(os.environ.get('DEBRIS_CLOSURE_BOOTSTRAP_DRAWS', '5000'))
CHECKPOINT_EVERY = int(os.environ.get('DEBRIS_CLOSURE_CHECKPOINT_EVERY', '10'))
FORCE_RECOMPUTE = os.environ.get('DEBRIS_CLOSURE_FORCE', '0') == '1'
MASTER_SEED = 20260830
DWELL_HOURS = 1.0

SIZE_CASES = {
    'gt_1mm': {'suffix': '30', 'old_column': 'window_contribution_gt_1mm_m2_yr', 'label': '>1 mm'},
    'gt_3p16mm': {'suffix': '35', 'old_column': 'window_contribution_gt_3p16mm_m2_yr', 'label': '>3.16 mm'},
    'gt_1cm': {'suffix': '40', 'old_column': 'window_contribution_gt_1cm_m2_yr', 'label': '>1 cm'},
    'gt_3p16cm': {'suffix': '45', 'old_column': 'window_contribution_gt_3p16cm_m2_yr', 'label': '>3.16 cm'},
    'gt_10cm': {'suffix': '50', 'old_column': 'window_contribution_gt_10cm_m2_yr', 'label': '>10 cm'},
}

print('research root:', PROJECT_ROOT)
print('Primary / sensitivity worlds:', N_PRIMARY, '/', N_SENSITIVITY)
print('Strict SBC worlds / posterior draws:', N_SBC, '/', SBC_DRAWS)


## 1. Frozen input and primary-analysis audit


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, 'rb') as handle:
        while True:
            block = handle.read(chunk_size)
            if not block:
                break
            digest.update(block)
    return digest.hexdigest()


def stable_seed(*parts):
    payload = '|'.join(map(str, parts)).encode()
    return int.from_bytes(hashlib.sha256(payload).digest()[:8], 'little') % (2**32 - 1)


required = [
    REFINEMENT_ZIP, OLD_RESPONSE_PATH, FINE_RESPONSE_PATH, EVENT_PATH, SELECTED_DETECTOR_PATH,
    DETECTOR_FOLDS_PATH, DETECTOR_RANKING_PATH, CATALOG_RECOVERY_PATH,
    CATALOG_SUMMARY_PATH,
]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing completed project inputs:\n' + '\n'.join(missing))
if not PREVIOUS_CONFIG_CANDIDATES:
    raise FileNotFoundError('No completed fine-grid final_experiment_config.json was found.')
preferred_previous_config = (
    PROJECT_ROOT / 'phase3_outputs' / 'final_bayesian_policy'
    / 'fine_grid_4fa819596ffc' / 'final_experiment_config.json'
)
PREVIOUS_CONFIG_PATH = (
    preferred_previous_config if preferred_previous_config.is_file()
    else PREVIOUS_CONFIG_CANDIDATES[-1]
)

input_hashes = {path.name: sha256_file(path) for path in required + [PREVIOUS_CONFIG_PATH]}
if input_hashes[REFINEMENT_ZIP.name] != EXPECTED_REFINEMENT_SHA256:
    raise RuntimeError('The completed b982 ORDEM ZIP checksum changed.')

events = pd.read_csv(EVENT_PATH)
detector = json.loads(SELECTED_DETECTOR_PATH.read_text())
detector_folds = pd.read_csv(DETECTOR_FOLDS_PATH)
detector_ranking = pd.read_csv(DETECTOR_RANKING_PATH)
catalog_recovery = pd.read_csv(CATALOG_RECOVERY_PATH)
catalog_summary = json.loads(CATALOG_SUMMARY_PATH.read_text())
previous_config = json.loads(PREVIOUS_CONFIG_PATH.read_text())

event_counts = events.association_tier.value_counts().to_dict()
expected_counts = {'unresolved': 106, 'tier_a_strict': 46, 'tier_b_empirical': 3}
if len(events) != 155 or event_counts != expected_counts:
    raise RuntimeError(f'Frozen event decomposition changed: {len(events)}, {event_counts}')

FALSE_RATE_PER_HOUR = float(detector['selection']['false_tracks_per_hour_upper95'])
UNRESOLVED_RATE_PER_HOUR = event_counts['unresolved'] / 24.0
NET_UNRESOLVED_SIGNAL_RATE = max(UNRESOLVED_RATE_PER_HOUR - FALSE_RATE_PER_HOUR, 0.0)
INFERENCE_DISPERSION = float(previous_config['negative_binomial_inference_dispersion'])
PRIOR_LOG_SD = float(previous_config['prior_log_sd'])
PRIOR_CORRELATION_LENGTH_KM = float(previous_config['prior_correlation_length_km'])
PRIOR_NUGGET_FRACTION = float(previous_config['prior_nugget_fraction'])

primary_analysis_spec = {
    'frozen_before_reviewer_closure_simulation': True,
    'primary_question': (
        'Does equal-weighted A-optimal adaptive scheduling reduce final smooth-population '
        'uncertainty versus the best equal-weighted fixed elevation under equal 24-hour time?'
    ),
    'primary_policy': 'unweighted_aopt',
    'primary_comparator': 'best_fixed_equal',
    'primary_endpoint': 'equal_weighted_posterior_variance_fraction_after_24_observations',
    'primary_estimand': 'paired ratio-of-means percent reduction across complete simulated worlds',
    'primary_success_rule': 'paired bootstrap 95% CI lower bound is greater than 0%',
    'primary_size_sensitivity_case': '>1 mm cumulative ORDEM response',
    'primary_event_stream': '106 unresolved events only; 46 strict and 3 supported associations excluded',
    'false_event_rate': '95% upper bound from the frozen real-data null experiment',
    'response_uncertainty': 'persistent ORDEM action-level nuisance parameter',
    'target_weights': 'equal across ten altitude shells',
    'satellite_weighting_role': 'not used in primary analysis; deferred extension',
    'secondary_confirmatory_checks': [
        'adaptive versus random uncertainty',
        'RMSE noninferiority to random with a 5% margin',
        '95% interval coverage between 90% and 98%',
    ],
    'sensitivity_only': [
        'other ORDEM size thresholds', '50% and 75% hidden-signal fractions',
        '25% and 50% broad-response contamination', 'five-mode latent basis',
        '70-85 degree action domain', 'localized anomaly', 'neighbor mixing',
    ],
    'independent_unit': 'complete simulated debris world',
    'primary_worlds': N_PRIMARY,
    'sensitivity_worlds_per_case': N_SENSITIVITY,
    'master_seed': MASTER_SEED,
}

config_payload = {
    'primary_analysis_spec': primary_analysis_spec,
    'input_hashes': input_hashes,
    'budget': BUDGET,
    'n_sbc': N_SBC,
    'sbc_burn': SBC_BURN,
    'sbc_draws': SBC_DRAWS,
    'sbc_thin': SBC_THIN,
}
config_hash = hashlib.sha256(json.dumps(config_payload, sort_keys=True).encode()).hexdigest()[:12]
OUTPUT_DIR = PROJECT_ROOT / 'phase3_outputs' / 'reviewer_closure' / f'closure_{config_hash}'
CHECKPOINT_DIR = OUTPUT_DIR / 'checkpoints'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'primary_analysis_spec.json').write_text(json.dumps(primary_analysis_spec, indent=2))

print('INPUT AUDIT PASSED')
print('  Events: 46 strict + 3 supported + 106 unresolved')
print('  Primary unresolved rate/hour:', round(UNRESOLVED_RATE_PER_HOUR, 4))
print('  False-event upper rate/hour:', round(FALSE_RATE_PER_HOUR, 4))
print('  Output:', OUTPUT_DIR)

short_rule_ranking = detector_ranking.loc[
    detector_ranking.track_rule.eq('short_0p8')
    & detector_ranking.config_id.isin(['ca_1.75', 'ca_2'])
].copy() if 'track_rule' in detector_ranking.columns else detector_ranking.loc[
    detector_ranking.config_id.isin(['ca_1.75', 'ca_2'])
].copy()
detector_bridge_summary = {
    'selected_detector': {
        'config_id': detector['selection']['config_id'],
        'method': detector['selection']['method'],
        'threshold': detector['selection']['threshold'],
        'track_rule': detector['selection']['track_rule'],
        'swerling_all_detection_probability': detector['selection'].get('pd_swerling_all'),
        'short_weak_detection_probability': detector['selection'].get('pd_focus_short_weak'),
        'target_trials': detector['selection'].get('target_trials'),
        'null_exposure_hours': detector['selection'].get('null_exposure_hours'),
        'false_tracks_per_24h_upper95': detector['selection'].get('false_tracks_per_24h_upper95'),
    },
    'heldout_detector_choice_counts': {
        str(key): int(value) for key, value in detector_folds.chosen_config_id.value_counts().items()
    },
    'ca_1p75_vs_ca_2_injection_sensitivity': short_rule_ranking.to_dict(orient='records'),
    'event_decomposition': event_counts,
    'hidden_population_rate_model': {
        'unresolved_events': int(event_counts['unresolved']),
        'unresolved_events_per_hour': UNRESOLVED_RATE_PER_HOUR,
        'false_event_upper_rate_per_hour': FALSE_RATE_PER_HOUR,
        'net_primary_signal_rate_per_hour': NET_UNRESOLVED_SIGNAL_RATE,
        'strict_and_supported_catalog_events_excluded': int(
            event_counts['tier_a_strict'] + event_counts['tier_b_empirical']
        ),
    },
    'catalog_first_validation': catalog_summary,
    'interpretation': (
        'Observed unresolved count rate calibrates the detected-event scale. ORDEM supplies the '
        'altitude/elevation response shape. This is an empirical bridge, not a diameter-to-Pd '
        'or measured side-lobe antenna model.'
    ),
}
(OUTPUT_DIR / 'detector_to_bayesian_bridge_summary.json').write_text(
    json.dumps(detector_bridge_summary, indent=2)
)


## 2. Rebuild the 49-elevation response at five existing ORDEM size thresholds


In [ ]:
def safe_extract(zip_path, destination):
    destination = Path(destination).resolve()
    destination.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path) as archive:
        for member in archive.infolist():
            target = (destination / member.filename).resolve()
            if destination != target and destination not in target.parents:
                raise RuntimeError(f'Unsafe ZIP member: {member.filename}')
        archive.extractall(destination)


def read_ordem_table(path):
    lines = Path(path).read_text(errors='replace').splitlines()
    header_index = next(i for i, line in enumerate(lines) if line.lstrip().startswith('Element'))
    columns = lines[header_index].split()
    rows = []
    for line in lines[header_index + 1:]:
        fields = line.split()
        if len(fields) != len(columns):
            continue
        try:
            rows.append([float(value) for value in fields])
        except ValueError:
            continue
    frame = pd.DataFrame(rows, columns=columns)
    if frame.empty:
        raise ValueError(f'No numeric ORDEM table parsed from {path}')
    frame['Element'] = frame.Element.astype(int)
    return frame


def parse_elevation(path):
    for line in Path(path).read_text(errors='replace').splitlines():
        if '!' not in line:
            continue
        raw, comment = line.split('!', 1)
        if comment.strip().lower().startswith('elevation'):
            return float(raw.strip())
    raise ValueError(f'No elevation in {path}')


def overlap_fraction(low, high):
    low = np.asarray(low, dtype=float)
    high = np.asarray(high, dtype=float)
    overlap = np.maximum(0.0, np.minimum(high, RANGE_MAX_KM) - np.maximum(low, RANGE_MIN_KM))
    return np.divide(overlap, high - low, out=np.zeros_like(overlap), where=high > low)


old_response = pd.read_csv(OLD_RESPONSE_PATH)
old_centerline = old_response.loc[np.isclose(old_response.azimuth_deg, 90.0)].copy()
old_expected = np.arange(65.0, 85.0 + 0.01, 2.5)
if set(np.round(old_centerline.elevation_deg.unique(), 6)) != set(np.round(old_expected, 6)):
    raise RuntimeError('The old response table lacks the nine exact azimuth-90 elevations.')

response_by_size = {key: {} for key in SIZE_CASES}
relative_sd_by_size = {key: {} for key in SIZE_CASES}

for size_key, spec in SIZE_CASES.items():
    for elevation, group in old_centerline.groupby('elevation_deg'):
        mids = 0.5 * (group.alt_low.to_numpy() + group.alt_high.to_numpy())
        bins = np.searchsorted(ALTITUDE_EDGES, mids, side='right') - 1
        values = group[spec['old_column']].to_numpy(dtype=float)
        response_by_size[size_key][float(elevation)] = np.asarray([
            values[bins == index].sum() for index in range(N_SHELLS)
        ])

with tempfile.TemporaryDirectory(prefix='debris_closure_ordem_') as temporary:
    safe_extract(REFINEMENT_ZIP, temporary)
    result_dirs = sorted({path.parent for path in Path(temporary).rglob('ORDEM.IN')})
    if len(result_dirs) != 40:
        raise RuntimeError(f'Expected 40 b982 result directories; found {len(result_dirs)}')
    for directory in result_dirs:
        if 'Complete' not in (directory / 'status.txt').read_text(errors='replace'):
            raise RuntimeError(f'Incomplete ORDEM result: {directory.name}')
        elevation = parse_elevation(directory / 'ORDEM.IN')
        flux = read_ordem_table(directory / 'IGLOOFLUX_TEL.OUT')
        sigma_range = read_ordem_table(directory / 'IGLOOFLUX_SIGMARAN_TEL.OUT')
        sigma_population = read_ordem_table(directory / 'IGLOOFLUX_SIGMAPOP_TEL.OUT')
        overlap = overlap_fraction(flux.rng_low, flux.rng_high)
        mids = 0.5 * (flux.alt_low.to_numpy() + flux.alt_high.to_numpy())
        bins = np.searchsorted(ALTITUDE_EDGES, mids, side='right') - 1
        for size_key, spec in SIZE_CASES.items():
            flux_columns = [c for c in flux if c.startswith('Flux_') and c.endswith(spec['suffix'])]
            range_columns = [f"Sig(ran)_{c.split('_', 1)[1]}" for c in flux_columns]
            population_columns = [f"Sig(pop)_{c.split('_', 1)[1]}" for c in flux_columns]
            if len(flux_columns) != 5:
                raise RuntimeError(f'{directory.name}/{size_key}: expected five material populations')
            contribution = flux[flux_columns].sum(axis=1).to_numpy() * overlap
            row = np.asarray([contribution[bins == index].sum() for index in range(N_SHELLS)])
            response_by_size[size_key][float(elevation)] = row
            element_range_sd = np.sqrt(np.square(sigma_range[range_columns].to_numpy()).sum(axis=1)) * overlap
            element_population_sd = np.sqrt(np.square(sigma_population[population_columns].to_numpy()).sum(axis=1)) * overlap
            total_sd = float(np.sqrt(np.square(np.hypot(element_range_sd, element_population_sd)).sum()))
            relative_sd_by_size[size_key][float(elevation)] = total_sd / max(float(contribution.sum()), 1e-30)

H_BY_SIZE = {}
RESPONSE_LOG_SD_BY_SIZE = {}
tidy_response_rows = []
identifiability_rows = []
for size_key, spec in SIZE_CASES.items():
    available_elevations = np.asarray(list(response_by_size[size_key].keys()), dtype=float)
    if set(np.round(available_elevations, 6)) != set(np.round(ELEVATIONS, 6)):
        missing_elevations = sorted(set(np.round(ELEVATIONS, 6)) - set(np.round(available_elevations, 6)))
        raise RuntimeError(f'{size_key}: missing elevations {missing_elevations}')
    matrix = np.stack([response_by_size[size_key][float(elevation)] for elevation in ELEVATIONS])
    if not np.isfinite(matrix).all() or (matrix < 0).any() or np.any(matrix.sum(axis=1) <= 0):
        raise RuntimeError(f'Invalid response matrix for {size_key}')
    H_BY_SIZE[size_key] = matrix
    known_elevations = np.asarray(sorted(relative_sd_by_size[size_key]), dtype=float)
    known_sd = np.asarray([relative_sd_by_size[size_key][value] for value in known_elevations])
    interpolated_sd = np.interp(ELEVATIONS, known_elevations, known_sd)
    RESPONSE_LOG_SD_BY_SIZE[size_key] = np.sqrt(np.log1p(np.square(interpolated_sd)))
    normalized = matrix / matrix.sum(axis=1, keepdims=True)
    singular_values = np.linalg.svd(normalized, compute_uv=False)
    cumulative = np.cumsum(np.square(singular_values)) / np.square(singular_values).sum()
    effective_rank = float(np.square(np.square(singular_values).sum()) / np.power(singular_values, 4).sum())
    for component, (value, fraction) in enumerate(zip(singular_values, cumulative), start=1):
        identifiability_rows.append({
            'size_case': size_key, 'size_label': spec['label'], 'component': component,
            'singular_value': value, 'cumulative_energy_fraction': fraction,
            'effective_rank': effective_rank,
            'components_for_99_percent': int(np.searchsorted(cumulative, 0.99) + 1),
        })
    for action, elevation in enumerate(ELEVATIONS):
        for shell, altitude in enumerate(ALTITUDE_CENTERS):
            tidy_response_rows.append({
                'size_case': size_key, 'size_label': spec['label'],
                'elevation_deg': elevation, 'altitude_center_km': altitude,
                'response_m2_yr': matrix[action, shell],
                'response_share_within_action': normalized[action, shell],
                'response_log_sd': RESPONSE_LOG_SD_BY_SIZE[size_key][action],
            })

size_response = pd.DataFrame(tidy_response_rows)
identifiability = pd.DataFrame(identifiability_rows)

                                                                                
frozen_fine_response = pd.read_csv(FINE_RESPONSE_PATH)
frozen_shell_columns = [column for column in frozen_fine_response if column.startswith('shell_')]
frozen_h = frozen_fine_response[frozen_shell_columns].to_numpy(dtype=float)
if frozen_h.shape != (N_ACTIONS, N_SHELLS):
    raise RuntimeError(f'Frozen fine response shape changed: {frozen_h.shape}')
if not np.allclose(
    frozen_fine_response.elevation_deg.to_numpy(dtype=float), ELEVATIONS,
    atol=1e-10, rtol=0.0,
):
    raise RuntimeError('Frozen fine response elevations changed.')
relative_rebuild_error = np.max(
    np.abs(H_BY_SIZE['gt_1mm'] - frozen_h) / np.maximum(np.abs(frozen_h), 1e-30)
)
absolute_rebuild_error = np.max(np.abs(H_BY_SIZE['gt_1mm'] - frozen_h))
if not np.allclose(H_BY_SIZE['gt_1mm'], frozen_h, rtol=2e-7, atol=1e-12):
    raise RuntimeError(
        f'Rebuilt >1 mm response does not reproduce the frozen matrix: '
        f'max relative {relative_rebuild_error:.3e}, absolute {absolute_rebuild_error:.3e}'
    )

size_response.to_csv(OUTPUT_DIR / 'size_specific_response_library_49x10.csv.gz', index=False, compression='gzip')
identifiability.to_csv(OUTPUT_DIR / 'response_identifiability_by_size.csv', index=False)

print('SIZE RESPONSE AUDIT PASSED')
print('  Rebuilt >1 mm maximum relative difference:', f'{relative_rebuild_error:.3e}')
for size_key, spec in SIZE_CASES.items():
    group = identifiability.loc[identifiability.size_case == size_key].iloc[0]
    print(f"  {spec['label']}: {int(group.components_for_99_percent)} components for 99%; effective rank {group.effective_rank:.2f}")


## 3. General response-aware inference model


In [ ]:
shell_distance = np.abs(ALTITUDE_CENTERS[:, None] - ALTITUDE_CENTERS[None, :])
scaled_distance = np.sqrt(3.0) * shell_distance / PRIOR_CORRELATION_LENGTH_KM
matern32 = (1.0 + scaled_distance) * np.exp(-scaled_distance)
K_X = PRIOR_LOG_SD**2 * (
    (1.0 - PRIOR_NUGGET_FRACTION) * matern32
    + PRIOR_NUGGET_FRACTION * np.eye(N_SHELLS)
)
K_X += np.eye(N_SHELLS) * 1e-8
EQUAL_WEIGHTS = np.repeat(1.0 / N_SHELLS, N_SHELLS)

eigenvalues, eigenvectors = np.linalg.eigh(K_X)
order = np.argsort(eigenvalues)[::-1]
eigenvalues = eigenvalues[order]
eigenvectors = eigenvectors[:, order]
KL5_BASIS = eigenvectors[:, :5]
KL5_COVARIANCE = np.diag(eigenvalues[:5])
KL5_PRIOR_VARIANCE_FRACTION = float(eigenvalues[:5].sum() / eigenvalues.sum())


@dataclass
class InferenceModel:
    condition_name: str
    h: np.ndarray
    basis: np.ndarray
    latent_covariance: np.ndarray
    k_q: np.ndarray
    allowed_actions: np.ndarray
    signal_rate_center: float
    background_rate: float
    hidden_fraction: float
    side_lobe_fraction: float
    size_case: str
    basis_name: str
    prior_covariance: np.ndarray | None = None
    prior_cholesky: np.ndarray | None = None
    best_fixed_action: int | None = None
    fixed_design_variance: np.ndarray | None = None

    def __post_init__(self):
        m = self.basis.shape[1]
        prior = np.zeros((m + N_ACTIONS, m + N_ACTIONS))
        prior[:m, :m] = self.latent_covariance
        prior[m:, m:] = self.k_q
        self.prior_covariance = prior
        self.prior_cholesky = np.linalg.cholesky(prior)

    @property
    def n_modes(self):
        return self.basis.shape[1]

    @property
    def n_state(self):
        return self.n_modes + N_ACTIONS

    @property
    def response_correction(self):
        return 0.5 * np.diag(self.k_q)

    @property
    def signal_scale(self):
        return self.signal_rate_center * DWELL_HOURS / self.h[CENTER_ACTION].sum()

    @property
    def x_prior_covariance(self):
        return self.basis @ self.latent_covariance @ self.basis.T


def response_covariance(size_case, correlation_deg=1.5):
    action_distance = np.abs(ELEVATIONS[:, None] - ELEVATIONS[None, :])
    correlation = 0.90 * np.exp(-0.5 * np.square(action_distance / correlation_deg)) + 0.10 * np.eye(N_ACTIONS)
    sd = RESPONSE_LOG_SD_BY_SIZE[size_case]
    covariance = sd[:, None] * correlation * sd[None, :]
    covariance += np.eye(N_ACTIONS) * 1e-8
    return covariance


def make_model(spec):
    size_case = spec.get('size_case', 'gt_1mm')
    basis_name = spec.get('basis', 'full10')
    if basis_name == 'full10':
        basis, latent_covariance = np.eye(N_SHELLS), K_X
    elif basis_name == 'kl5':
        basis, latent_covariance = KL5_BASIS, KL5_COVARIANCE
    else:
        raise ValueError(basis_name)
    hidden_fraction = float(spec.get('hidden_fraction', 1.0))
    signal_rate = hidden_fraction * NET_UNRESOLVED_SIGNAL_RATE
    background_rate = FALSE_RATE_PER_HOUR + (1.0 - hidden_fraction) * NET_UNRESOLVED_SIGNAL_RATE
    domain = spec.get('action_domain', '65_89')
    if domain == '65_89':
        allowed = np.arange(N_ACTIONS)
    elif domain == '70_85':
        allowed = np.where((ELEVATIONS >= 70.0) & (ELEVATIONS <= 85.0))[0]
    else:
        raise ValueError(domain)
    return InferenceModel(
        condition_name=spec['condition'], h=H_BY_SIZE[size_case].copy(), basis=basis,
        latent_covariance=latent_covariance.copy(), k_q=response_covariance(size_case),
        allowed_actions=allowed, signal_rate_center=signal_rate,
        background_rate=background_rate, hidden_fraction=hidden_fraction,
        side_lobe_fraction=float(spec.get('side_lobe_fraction', 0.0)),
        size_case=size_case, basis_name=basis_name,
    )


def state_x(model, state):
    return model.basis @ state[:model.n_modes]


def theta_moments(model, mean, covariance):
    x_mean = model.basis @ mean[:model.n_modes]
    x_covariance = model.basis @ covariance[:model.n_modes, :model.n_modes] @ model.basis.T
    diagonal = np.maximum(np.diag(x_covariance), 0.0)
    theta_mean = np.exp(x_mean + 0.5 * diagonal)
    theta_variance = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
    return x_mean, x_covariance, theta_mean, theta_variance


def observation_linearization(model, mean, covariance, action):
    x = state_x(model, mean)
    q = mean[model.n_modes + action]
    components = (
        model.signal_scale * np.exp(q - model.response_correction[action])
        * model.h[action] * np.exp(x)
    )
    signal = float(components.sum())
    expected = model.background_rate * DWELL_HOURS + signal
    gradient = np.zeros(model.n_state)
    gradient[:model.n_modes] = model.basis.T @ components
    gradient[model.n_modes + action] = signal
    variance = expected + expected**2 / INFERENCE_DISPERSION
    return expected, gradient, variance


def sequential_update(model, mean, covariance, action, observed, iterations=5):
    prior_mean = mean.copy()
    prior_covariance = covariance.copy()
    trial = prior_mean.copy()
    for _ in range(iterations):
        expected, gradient, observation_variance = observation_linearization(model, trial, prior_covariance, action)
        projected = prior_covariance @ gradient
        denominator = observation_variance + gradient @ projected
        adjusted = observed - expected + gradient @ (trial - prior_mean)
        proposal = prior_mean + projected * adjusted / max(denominator, 1e-10)
        trial = 0.35 * trial + 0.65 * proposal
    expected, gradient, observation_variance = observation_linearization(model, trial, prior_covariance, action)
    projected = prior_covariance @ gradient
    updated = prior_covariance - np.outer(projected, projected) / max(
        observation_variance + gradient @ projected, 1e-10
    )
    updated = 0.5 * (updated + updated.T)
    values, vectors = np.linalg.eigh(updated)
    updated = (vectors * np.maximum(values, 1e-10)) @ vectors.T
    return trial, updated


def batch_map_laplace(model, actions, counts, start_state):
    actions = np.asarray(actions, dtype=int)
    counts = np.asarray(counts, dtype=float)
    observed_h = model.h[actions]
    corrections = model.response_correction[actions]
    cholesky = model.prior_cholesky

    def objective_gradient(whitened):
        state = cholesky @ whitened
        x = state_x(model, state)
        q = state[model.n_modes:][actions]
        components = (
            model.signal_scale * np.exp(q - corrections)[:, None]
            * observed_h * np.exp(x)[None, :]
        )
        signal = components.sum(axis=1)
        means = np.maximum(model.background_rate + signal, 1e-12)
        k = INFERENCE_DISPERSION
        log_likelihood = (
            gammaln(counts + k) - gammaln(k) - gammaln(counts + 1.0)
            + k * np.log(k) + counts * np.log(means) - (counts + k) * np.log(means + k)
        )
        derivative_nll_mean = k * (means - counts) / (means * (means + k))
        gradient_state = np.zeros(model.n_state)
        gradient_state[:model.n_modes] = model.basis.T @ (
            derivative_nll_mean[:, None] * components
        ).sum(axis=0)
        np.add.at(gradient_state[model.n_modes:], actions, derivative_nll_mean * signal)
        return float(0.5 * whitened @ whitened - log_likelihood.sum()), whitened + cholesky.T @ gradient_state

    initial = np.linalg.solve(cholesky, np.asarray(start_state, dtype=float))
    result = minimize(
        objective_gradient, initial, jac=True, method='L-BFGS-B',
        bounds=[(-8.0, 8.0)] * model.n_state,
        options={'maxiter': 200, 'maxls': 40, 'ftol': 1e-10, 'gtol': 2e-6},
    )
    state = cholesky @ result.x
    x = state_x(model, state)
    q = state[model.n_modes:][actions]
    components = (
        model.signal_scale * np.exp(q - corrections)[:, None]
        * observed_h * np.exp(x)[None, :]
    )
    signal = components.sum(axis=1)
    means = np.maximum(model.background_rate + signal, 1e-12)
    gradient_state = np.zeros((len(actions), model.n_state))
    gradient_state[:, :model.n_modes] = components @ model.basis
    gradient_state[np.arange(len(actions)), model.n_modes + actions] = signal
    gradient_white = gradient_state @ cholesky
    observation_variance = means + means**2 / INFERENCE_DISPERSION
    precision_white = np.eye(model.n_state) + gradient_white.T @ (
        gradient_white / observation_variance[:, None]
    )
    covariance = cholesky @ np.linalg.inv(precision_white) @ cholesky.T
    covariance = 0.5 * (covariance + covariance.T)
    gradient_norm = float(np.linalg.norm(result.jac, ord=np.inf))
    success = bool(result.success or (np.isfinite(result.fun) and gradient_norm < 1e-4))
    return state, covariance, success, int(result.nit), gradient_norm


def posterior_metrics(model, mean, covariance, truth_x):
    x_mean, x_covariance, theta_mean, theta_variance = theta_moments(model, mean, covariance)
    truth_theta = np.exp(truth_x)
    x_sd = np.sqrt(np.maximum(np.diag(x_covariance), 0.0))
    low95 = np.exp(x_mean - 1.96 * x_sd)
    high95 = np.exp(x_mean + 1.96 * x_sd)
    prior_mean = np.zeros(model.n_state)
    _, _, _, initial_theta_variance = theta_moments(model, prior_mean, model.prior_covariance)
    return {
        'equal_uncertainty_fraction': float(
            np.sum(EQUAL_WEIGHTS * theta_variance) / np.sum(EQUAL_WEIGHTS * initial_theta_variance)
        ),
        'equal_weighted_rmse': float(np.sqrt(np.sum(EQUAL_WEIGHTS * np.square(theta_mean - truth_theta)))),
        'cellwise_95_coverage': float(np.mean((truth_theta >= low95) & (truth_theta <= high95))),
        'aggregate_log_mean': float(EQUAL_WEIGHTS @ x_mean),
        'aggregate_log_sd': float(np.sqrt(max(EQUAL_WEIGHTS @ x_covariance @ EQUAL_WEIGHTS, 1e-12))),
    }


def aopt_scores(model, mean, covariance):
    _, _, _, before = theta_moments(model, mean, covariance)
    x_mean, _, _, _ = theta_moments(model, mean, covariance)
    scores = np.full(N_ACTIONS, -np.inf)
    for action in model.allowed_actions:
        _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
        projected = covariance @ gradient
        updated = covariance - np.outer(projected, projected) / max(
            observation_variance + gradient @ projected, 1e-10
        )
        x_covariance = model.basis @ updated[:model.n_modes, :model.n_modes] @ model.basis.T
        diagonal = np.maximum(np.diag(x_covariance), 0.0)
        after = (np.exp(diagonal) - 1.0) * np.exp(2.0 * x_mean + diagonal)
        scores[action] = np.sum(EQUAL_WEIGHTS * (before - after))
    return scores


def eig_scores(model, mean, covariance):
    scores = np.full(N_ACTIONS, -np.inf)
    for action in model.allowed_actions:
        _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
        ratio = gradient @ covariance @ gradient / observation_variance
        scores[action] = 0.5 * np.log1p(max(ratio, 0.0))
    return scores


def select_best_fixed(model):
    predicted = np.full(N_ACTIONS, np.inf)
    for action in model.allowed_actions:
        mean = np.zeros(model.n_state)
        covariance = model.prior_covariance.copy()
        for _ in range(BUDGET):
            _, gradient, observation_variance = observation_linearization(model, mean, covariance, action)
            projected = covariance @ gradient
            covariance -= np.outer(projected, projected) / max(
                observation_variance + gradient @ projected, 1e-10
            )
            covariance = 0.5 * (covariance + covariance.T)
        predicted[action] = posterior_metrics(model, mean, covariance, np.zeros(N_SHELLS))['equal_uncertainty_fraction']
    model.best_fixed_action = int(np.argmin(predicted))
    model.fixed_design_variance = predicted


def choose_action(model, policy, mean, covariance, rng):
    if policy == 'best_fixed_equal':
        return model.best_fixed_action
    if policy == 'random':
        return int(rng.choice(model.allowed_actions))
    if policy == 'unweighted_aopt':
        return int(np.argmax(aopt_scores(model, mean, covariance)))
    if policy == 'eig':
        return int(np.argmax(eig_scores(model, mean, covariance)))
    raise ValueError(policy)

print('MODEL CODE READY')
print('  Five KL modes capture', f'{100 * KL5_PRIOR_VARIANCE_FRACTION:.1f}%', 'of prior log-field variance')


## 4. Reviewer-closure experiment


In [ ]:
POLICIES = ('best_fixed_equal', 'random', 'unweighted_aopt', 'eig')
POLICY_LABELS = {
    'best_fixed_equal': 'Best fixed',
    'random': 'Random',
    'unweighted_aopt': 'Adaptive A-opt',
    'eig': 'Adaptive information gain',
}
POLICY_COLORS = {
    'best_fixed_equal': '#595959',
    'random': '#4C78A8',
    'unweighted_aopt': '#E45756',
    'eig': '#54A24B',
}

CONDITION_SPECS = [
    {
        'condition': 'primary_unresolved_gt1mm', 'display_label': 'Primary: >1 mm, unresolved only',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'model_matched',
        'analysis_role': 'primary', 'n_replicates': N_PRIMARY,
    },
    *[
        {
            'condition': f'size_{size_case}', 'display_label': f"Size response {SIZE_CASES[size_case]['label']}",
            'size_case': size_case, 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
            'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'model_matched',
            'analysis_role': 'size_sensitivity', 'n_replicates': N_SENSITIVITY,
        }
        for size_case in ('gt_3p16mm', 'gt_1cm', 'gt_3p16cm', 'gt_10cm')
    ],
    {
        'condition': 'hidden_fraction_0p75', 'display_label': '75% of unresolved rate is target signal',
        'size_case': 'gt_1mm', 'hidden_fraction': 0.75, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'model_matched',
        'analysis_role': 'contamination_sensitivity', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'hidden_fraction_0p50', 'display_label': '50% of unresolved rate is target signal',
        'size_case': 'gt_1mm', 'hidden_fraction': 0.50, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'model_matched',
        'analysis_role': 'contamination_sensitivity', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'broad_response_0p25', 'display_label': '25% unmodeled broad response',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.25,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'model_matched',
        'analysis_role': 'side_lobe_sensitivity', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'broad_response_0p50', 'display_label': '50% unmodeled broad response',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.50,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'model_matched',
        'analysis_role': 'side_lobe_sensitivity', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'latent_basis_kl5', 'display_label': 'Five-mode smooth latent field',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
        'basis': 'kl5', 'action_domain': '65_89', 'population_scenario': 'model_matched',
        'analysis_role': 'identifiability_sensitivity', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'action_domain_70_85', 'display_label': 'Actions restricted to 70-85 deg',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '70_85', 'population_scenario': 'model_matched',
        'analysis_role': 'action_domain_sensitivity', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'localized_population_anomaly', 'display_label': 'Localized population anomaly',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'localized_anomaly',
        'analysis_role': 'population_misspecification', 'n_replicates': N_SENSITIVITY,
    },
    {
        'condition': 'neighbor_shell_mixing', 'display_label': '15% neighboring-shell mixing',
        'size_case': 'gt_1mm', 'hidden_fraction': 1.0, 'side_lobe_fraction': 0.0,
        'basis': 'full10', 'action_domain': '65_89', 'population_scenario': 'neighbor_mixing',
        'analysis_role': 'population_misspecification', 'n_replicates': N_SENSITIVITY,
    },
]


def neighbor_mixing_matrix(fraction=0.15):
    matrix = np.zeros((N_SHELLS, N_SHELLS))
    for source in range(N_SHELLS):
        neighbors = []
        if source > 0:
            neighbors.append(source - 1)
        if source < N_SHELLS - 1:
            neighbors.append(source + 1)
        matrix[source, source] = 1.0 - fraction
        for destination in neighbors:
            matrix[destination, source] += fraction / len(neighbors)
    return matrix


MIXING_MATRIX = neighbor_mixing_matrix(0.15)


def truth_and_streams(spec, model, replicate):
    rng = np.random.default_rng(stable_seed(MASTER_SEED, 'truth', spec['condition'], replicate))
    truth_x = rng.multivariate_normal(np.zeros(N_SHELLS), K_X)
    truth_q = rng.multivariate_normal(np.zeros(N_ACTIONS), model.k_q)
    metadata = {'anomaly_center_km': np.nan, 'truth_type': 'model_matched'}
    scenario = spec['population_scenario']
    if scenario == 'localized_anomaly':
        center = int(rng.integers(N_SHELLS))
        width = float(rng.uniform(8.0, 15.0))
        amplitude = float(rng.uniform(np.log(2.0), np.log(3.0)))
        truth_x = truth_x + amplitude * np.exp(
            -0.5 * np.square((ALTITUDE_CENTERS - ALTITUDE_CENTERS[center]) / width)
        )
        metadata = {'anomaly_center_km': float(ALTITUDE_CENTERS[center]), 'truth_type': 'localized_anomaly'}
    elif scenario == 'neighbor_mixing':
        truth_x = np.log(np.maximum(MIXING_MATRIX @ np.exp(truth_x), 1e-12))
        metadata = {'anomaly_center_km': np.nan, 'truth_type': 'neighbor_mixing'}
    elif scenario != 'model_matched':
        raise ValueError(scenario)

    nominal_h = model.h
    shell_shape = nominal_h.sum(axis=0)
    shell_shape = shell_shape / shell_shape.sum()
    broad_h = nominal_h.sum(axis=1)[:, None] * shell_shape[None, :]
    truth_h = (1.0 - model.side_lobe_fraction) * nominal_h + model.side_lobe_fraction * broad_h
    truth_h = truth_h * np.exp(
        truth_q[:, None] - model.response_correction[:, None]
    )
    means = model.background_rate + model.signal_scale * (truth_h @ np.exp(truth_x))
    count_rng = np.random.default_rng(stable_seed(MASTER_SEED, 'counts', spec['condition'], replicate))
    probability = INFERENCE_DISPERSION / (INFERENCE_DISPERSION + means[:, None])
    streams = count_rng.negative_binomial(
        INFERENCE_DISPERSION, probability, size=(N_ACTIONS, BUDGET)
    )
    return truth_x, truth_q, streams, means, metadata


def run_policy(spec, model, policy, replicate, truth_x, streams):
    rng = np.random.default_rng(stable_seed(MASTER_SEED, 'policy', spec['condition'], policy, replicate))
    mean = np.zeros(model.n_state)
    covariance = model.prior_covariance.copy()
    visits = np.zeros(N_ACTIONS, dtype=int)
    history_actions, history_counts = [], []
    step_rows, action_rows = [], []

    initial = posterior_metrics(model, mean, covariance, truth_x)
    step_rows.append({
        **{k: spec[k] for k in ('condition', 'display_label', 'analysis_role', 'size_case',
                                'hidden_fraction', 'side_lobe_fraction', 'basis', 'action_domain',
                                'population_scenario')},
        'replicate': replicate, 'policy': policy, 'step': 0, **initial,
        'posterior_method': 'prior', 'cumulative_events': 0,
    })

    for step in range(1, BUDGET + 1):
        action = choose_action(model, policy, mean, covariance, rng)
        visit = int(visits[action])
        observed = int(streams[action, visit])
        visits[action] += 1
        predicted = observation_linearization(model, mean, covariance, action)[0]
        history_actions.append(action)
        history_counts.append(observed)
        mean, covariance = sequential_update(model, mean, covariance, action, observed)
        metrics = posterior_metrics(model, mean, covariance, truth_x)
        step_rows.append({
            **{k: spec[k] for k in ('condition', 'display_label', 'analysis_role', 'size_case',
                                    'hidden_fraction', 'side_lobe_fraction', 'basis', 'action_domain',
                                    'population_scenario')},
            'replicate': replicate, 'policy': policy, 'step': step, **metrics,
            'posterior_method': 'sequential', 'cumulative_events': int(sum(history_counts)),
        })
        action_rows.append({
            'condition': spec['condition'], 'replicate': replicate, 'policy': policy,
            'step': step, 'action_index': action, 'elevation_deg': ELEVATIONS[action],
            'observed_events': observed, 'predicted_events_before_update': predicted,
            'visit_number': visit + 1,
        })

    final_mean, final_covariance, success, iterations, gradient_norm = batch_map_laplace(
        model, history_actions, history_counts, mean
    )
    final_metrics = posterior_metrics(model, final_mean, final_covariance, truth_x)
    step_rows[-1].update(final_metrics)
    step_rows[-1].update({
        'posterior_method': 'batch_map_fisher_laplace', 'optimizer_success': success,
        'optimizer_iterations': iterations, 'optimizer_gradient_inf_norm': gradient_norm,
    })
    return step_rows, action_rows, history_actions, history_counts, final_mean, final_covariance


condition_config = pd.DataFrame(CONDITION_SPECS)
condition_config.to_csv(OUTPUT_DIR / 'reviewer_closure_condition_registry.csv', index=False)
(OUTPUT_DIR / 'reviewer_closure_config.json').write_text(json.dumps(config_payload | {
    'conditions': CONDITION_SPECS,
    'policies': list(POLICIES),
    'kl5_prior_variance_fraction': KL5_PRIOR_VARIANCE_FRACTION,
}, indent=2))

all_step_frames = []
all_action_frames = []
simulation_started = time.time()

for condition_number, spec in enumerate(CONDITION_SPECS, start=1):
    model = make_model(spec)
    select_best_fixed(model)
    n_replicates = int(spec['n_replicates'])
    step_path = CHECKPOINT_DIR / f"steps_{spec['condition']}.csv.gz"
    action_path = CHECKPOINT_DIR / f"actions_{spec['condition']}.csv.gz"
    if FORCE_RECOMPUTE:
        for path in (step_path, action_path):
            if path.exists():
                path.unlink()
    existing_steps = pd.read_csv(step_path) if step_path.exists() else pd.DataFrame()
    existing_actions = pd.read_csv(action_path) if action_path.exists() else pd.DataFrame()
    if len(existing_actions):
        counts = existing_actions.groupby(['replicate', 'policy']).size()
        complete_worlds = set(
            counts[counts == BUDGET].reset_index().groupby('replicate')
            .filter(lambda frame: set(frame.policy) == set(POLICIES)).replicate.astype(int)
        )
    else:
        complete_worlds = set()
    print(
        f"\nCondition {condition_number}/{len(CONDITION_SPECS)}: {spec['display_label']}\n"
        f"  best fixed = {ELEVATIONS[model.best_fixed_action]:.1f} deg; resume {len(complete_worlds)}/{n_replicates}"
    )
    pending_steps, pending_actions = [], []
    since_flush = 0
    for replicate in range(n_replicates):
        if replicate in complete_worlds:
            continue
        truth_x, truth_q, streams, truth_means, metadata = truth_and_streams(spec, model, replicate)
        for policy in POLICIES:
            rows, actions, *_ = run_policy(spec, model, policy, replicate, truth_x, streams)
            for row in rows:
                row.update(metadata)
                row['best_fixed_elevation_deg'] = ELEVATIONS[model.best_fixed_action]
            for row in actions:
                row.update(metadata)
                row['best_fixed_elevation_deg'] = ELEVATIONS[model.best_fixed_action]
            pending_steps.extend(rows)
            pending_actions.extend(actions)
        since_flush += 1
        if since_flush >= CHECKPOINT_EVERY:
            existing_steps = pd.concat([existing_steps, pd.DataFrame(pending_steps)], ignore_index=True)
            existing_actions = pd.concat([existing_actions, pd.DataFrame(pending_actions)], ignore_index=True)
            existing_steps.to_csv(step_path, index=False, compression='gzip')
            existing_actions.to_csv(action_path, index=False, compression='gzip')
            pending_steps.clear(); pending_actions.clear(); since_flush = 0
            print(f'  checkpoint through world {replicate + 1}/{n_replicates}', flush=True)
    if pending_steps:
        existing_steps = pd.concat([existing_steps, pd.DataFrame(pending_steps)], ignore_index=True)
        existing_actions = pd.concat([existing_actions, pd.DataFrame(pending_actions)], ignore_index=True)
        existing_steps.to_csv(step_path, index=False, compression='gzip')
        existing_actions.to_csv(action_path, index=False, compression='gzip')
    expected_actions = n_replicates * len(POLICIES) * BUDGET
    if len(existing_actions) != expected_actions:
        raise RuntimeError(f"{spec['condition']}: {len(existing_actions)} action rows, expected {expected_actions}")
    all_step_frames.append(existing_steps)
    all_action_frames.append(existing_actions)

step_metrics = pd.concat(all_step_frames, ignore_index=True)
action_history = pd.concat(all_action_frames, ignore_index=True)
step_metrics.to_csv(OUTPUT_DIR / 'reviewer_closure_step_metrics.csv.gz', index=False, compression='gzip')
action_history.to_csv(OUTPUT_DIR / 'reviewer_closure_action_history.csv.gz', index=False, compression='gzip')

print('\nREVIEWER-CLOSURE SIMULATION COMPLETE')
print('  Step rows:', f'{len(step_metrics):,}')
print('  Action rows:', f'{len(action_history):,}')
print('  Wall time, minutes:', round((time.time() - simulation_started) / 60.0, 2))


## 5. Paired primary inference and sensitivity summaries


In [ ]:
final_metrics = step_metrics.loc[step_metrics.step == BUDGET].copy()


def bootstrap_ratio_improvement(candidate, baseline, seed, draws=BOOTSTRAP_DRAWS):
    candidate = np.asarray(candidate, dtype=float)
    baseline = np.asarray(baseline, dtype=float)
    if len(candidate) != len(baseline):
        raise ValueError('Paired samples must have equal length.')
    rng = np.random.default_rng(seed)
    indices = rng.integers(0, len(candidate), size=(draws, len(candidate)))
    sampled_candidate = candidate[indices].mean(axis=1)
    sampled_baseline = baseline[indices].mean(axis=1)
    values = 100.0 * (1.0 - sampled_candidate / sampled_baseline)
    point = 100.0 * (1.0 - candidate.mean() / baseline.mean())
    return point, float(np.quantile(values, 0.025)), float(np.quantile(values, 0.975))


summary_rows = []
for keys, group in final_metrics.groupby(['condition', 'display_label', 'analysis_role', 'policy']):
    condition, display_label, analysis_role, policy = keys
    summary_rows.append({
        'condition': condition, 'display_label': display_label, 'analysis_role': analysis_role,
        'policy': policy, 'n_worlds': group.replicate.nunique(),
        'equal_uncertainty_fraction_mean': group.equal_uncertainty_fraction.mean(),
        'equal_uncertainty_fraction_se': group.equal_uncertainty_fraction.std(ddof=1) / np.sqrt(len(group)),
        'equal_weighted_rmse_mean': group.equal_weighted_rmse.mean(),
        'cellwise_95_coverage_mean': group.cellwise_95_coverage.mean(),
        'optimizer_success_fraction': group.optimizer_success.eq(True).mean(),
        'best_fixed_elevation_deg': group.best_fixed_elevation_deg.iloc[0],
    })
final_summary = pd.DataFrame(summary_rows)

comparison_rows = []
for spec in CONDITION_SPECS:
    group = final_metrics.loc[final_metrics.condition == spec['condition']]
    candidate = group.loc[group.policy == 'unweighted_aopt'].set_index('replicate')
    for baseline_name in ('best_fixed_equal', 'random', 'eig'):
        baseline = group.loc[group.policy == baseline_name].set_index('replicate')
        shared = candidate.index.intersection(baseline.index)
        uncertainty = bootstrap_ratio_improvement(
            candidate.loc[shared, 'equal_uncertainty_fraction'],
            baseline.loc[shared, 'equal_uncertainty_fraction'],
            stable_seed(MASTER_SEED, 'bootstrap_uncertainty', spec['condition'], baseline_name),
        )
        rmse_difference = (
            candidate.loc[shared, 'equal_weighted_rmse'].to_numpy()
            - baseline.loc[shared, 'equal_weighted_rmse'].to_numpy()
        )
        rng = np.random.default_rng(stable_seed(MASTER_SEED, 'bootstrap_rmse', spec['condition'], baseline_name))
        indices = rng.integers(0, len(shared), size=(BOOTSTRAP_DRAWS, len(shared)))
        rmse_bootstrap = rmse_difference[indices].mean(axis=1)
        comparison_rows.append({
            'condition': spec['condition'], 'display_label': spec['display_label'],
            'analysis_role': spec['analysis_role'], 'baseline': baseline_name,
            'paired_worlds': len(shared),
            'uncertainty_improvement_percent': uncertainty[0],
            'uncertainty_improvement_ci95_low': uncertainty[1],
            'uncertainty_improvement_ci95_high': uncertainty[2],
            'rmse_candidate_minus_baseline': float(rmse_difference.mean()),
            'rmse_difference_ci95_low': float(np.quantile(rmse_bootstrap, 0.025)),
            'rmse_difference_ci95_high': float(np.quantile(rmse_bootstrap, 0.975)),
            'baseline_rmse_mean': float(baseline.loc[shared, 'equal_weighted_rmse'].mean()),
        })
paired_comparisons = pd.DataFrame(comparison_rows)

final_summary.to_csv(OUTPUT_DIR / 'reviewer_closure_final_summary.csv', index=False)
paired_comparisons.to_csv(OUTPUT_DIR / 'reviewer_closure_paired_comparisons.csv', index=False)

primary_summary = final_summary.loc[final_summary.condition == 'primary_unresolved_gt1mm']
primary_comparison = paired_comparisons.loc[
    (paired_comparisons.condition == 'primary_unresolved_gt1mm')
    & (paired_comparisons.baseline == 'best_fixed_equal')
].iloc[0]
display(primary_summary)
display(paired_comparisons.loc[paired_comparisons.baseline == 'best_fixed_equal'])


## 6. Strict prior-predictive calibration and posterior-sampling cross-check


In [ ]:
def log_likelihood_state(model, actions, counts, state):
    actions = np.asarray(actions, dtype=int)
    counts = np.asarray(counts, dtype=float)
    x = state_x(model, state)
    q = state[model.n_modes:][actions]
    signal = (
        model.signal_scale
        * np.exp(q - model.response_correction[actions])
        * (model.h[actions] @ np.exp(x))
    )
    means = np.maximum(model.background_rate + signal, 1e-12)
    k = INFERENCE_DISPERSION
    return float(np.sum(
        gammaln(counts + k) - gammaln(k) - gammaln(counts + 1.0)
        + k * np.log(k) + counts * np.log(means) - (counts + k) * np.log(means + k)
    ))


def elliptical_slice_step(model, actions, counts, current, current_ll, rng):
    direction = model.prior_cholesky @ rng.normal(size=model.n_state)
    log_threshold = current_ll + np.log(rng.uniform())
    angle = rng.uniform(0.0, 2.0 * np.pi)
    angle_min = angle - 2.0 * np.pi
    angle_max = angle
    for evaluations in range(1, 1001):
        proposal = current * np.cos(angle) + direction * np.sin(angle)
        proposal_ll = log_likelihood_state(model, actions, counts, proposal)
        if proposal_ll > log_threshold:
            return proposal, proposal_ll, evaluations
        if angle < 0:
            angle_min = angle
        else:
            angle_max = angle
        angle = rng.uniform(angle_min, angle_max)
    raise RuntimeError('Elliptical slice sampler exceeded 1000 bracket evaluations.')


def sample_elliptical_slice(model, actions, counts, initial, seed):
    rng = np.random.default_rng(seed)
    current = np.asarray(initial, dtype=float).copy()
    current_ll = log_likelihood_state(model, actions, counts, current)
    retained = []
    evaluations = []
    total_iterations = SBC_BURN + SBC_DRAWS * SBC_THIN
    for iteration in range(total_iterations):
        current, current_ll, used = elliptical_slice_step(
            model, actions, counts, current, current_ll, rng
        )
        evaluations.append(used)
        if iteration >= SBC_BURN and (iteration - SBC_BURN) % SBC_THIN == 0:
            retained.append(current.copy())
    return np.asarray(retained), float(np.mean(evaluations))


def dispersed_chain_initial(model, center, seed, scale=0.25):
    rng = np.random.default_rng(seed)
    return np.asarray(center) + scale * (model.prior_cholesky @ rng.normal(size=model.n_state))


def split_rhat(chains):
    chains = np.asarray(chains, dtype=float)
    half = chains.shape[1] // 2
    split = np.concatenate([chains[:, :half], chains[:, -half:]], axis=0)
    n = split.shape[1]
    within = np.mean(np.var(split, axis=1, ddof=1))
    between = n * np.var(np.mean(split, axis=1), ddof=1)
    variance = ((n - 1.0) / n) * within + between / n
    return float(np.sqrt(variance / max(within, 1e-15)))


def autocorrelation_ess(chains):
    chains = np.asarray(chains, dtype=float)
    m, n = chains.shape
    centered = chains - chains.mean(axis=1, keepdims=True)
    variances = np.sum(centered**2, axis=1)
    rho = []
    for lag in range(1, n):
        values = []
        for chain in range(m):
            if variances[chain] > 0:
                values.append(np.sum(centered[chain, :-lag] * centered[chain, lag:]) / variances[chain])
        rho.append(float(np.mean(values)) if values else 0.0)
    positive_sum = 0.0
    for index in range(0, len(rho) - 1, 2):
        pair = rho[index] + rho[index + 1]
        if pair < 0:
            break
        positive_sum += pair
    return float(min(m * n, m * n / max(1.0 + 2.0 * positive_sum, 1e-12)))


primary_spec = next(spec for spec in CONDITION_SPECS if spec['analysis_role'] == 'primary')
primary_model = make_model(primary_spec)
select_best_fixed(primary_model)
calibration_path = CHECKPOINT_DIR / 'strict_prior_predictive_sbc.csv'
if FORCE_RECOMPUTE and calibration_path.exists():
    calibration_path.unlink()
calibration_existing = pd.read_csv(calibration_path) if calibration_path.exists() else pd.DataFrame()
completed_sbc = set(calibration_existing.replicate.astype(int)) if len(calibration_existing) else set()
calibration_rows = []
calibration_started = time.time()

for replicate in range(N_SBC):
    if replicate in completed_sbc:
        continue
    truth_x, truth_q, streams, _, _ = truth_and_streams(primary_spec, primary_model, replicate)
    _, _, actions, counts, laplace_mean, laplace_covariance = run_policy(
        primary_spec, primary_model, 'unweighted_aopt', replicate, truth_x, streams
    )
    truth_aggregate = float(EQUAL_WEIGHTS @ truth_x)
    laplace_x_mean = state_x(primary_model, laplace_mean)
    laplace_x_covariance = (
        primary_model.basis
        @ laplace_covariance[:primary_model.n_modes, :primary_model.n_modes]
        @ primary_model.basis.T
    )
    laplace_aggregate_mean = float(EQUAL_WEIGHTS @ laplace_x_mean)
    laplace_aggregate_sd = float(np.sqrt(max(
        EQUAL_WEIGHTS @ laplace_x_covariance @ EQUAL_WEIGHTS, 1e-12
    )))
    laplace_rng = np.random.default_rng(stable_seed(MASTER_SEED, 'laplace_sbc', replicate))
    laplace_draws = laplace_rng.normal(laplace_aggregate_mean, laplace_aggregate_sd, size=SBC_DRAWS)

    chain0_initial = dispersed_chain_initial(
        primary_model, laplace_mean, stable_seed(MASTER_SEED, 'ess_initial', replicate, 0)
    )
    chain0, mean_evaluations = sample_elliptical_slice(
        primary_model, actions, counts, chain0_initial,
        stable_seed(MASTER_SEED, 'ess_sbc', replicate, 0),
    )
    chain0_x = chain0[:, :primary_model.n_modes] @ primary_model.basis.T
    aggregate_chain0 = chain0_x @ EQUAL_WEIGHTS
    rank_sampling = int(np.sum(aggregate_chain0 < truth_aggregate))
    rank_laplace = int(np.sum(laplace_draws < truth_aggregate))
    sampling_low, sampling_high = np.quantile(chain0_x, [0.025, 0.975], axis=0)
    sampling_cell_coverage = float(np.mean((truth_x >= sampling_low) & (truth_x <= sampling_high)))

    rhat_value = np.nan
    effective_draws = np.nan
    if replicate < N_RHAT_WORLDS:
        aggregate_chains = [aggregate_chain0]
        for chain_index in range(1, 4):
            chain_initial = dispersed_chain_initial(
                primary_model, laplace_mean,
                stable_seed(MASTER_SEED, 'ess_initial', replicate, chain_index),
            )
            chain, _ = sample_elliptical_slice(
                primary_model, actions, counts, chain_initial,
                stable_seed(MASTER_SEED, 'ess_sbc', replicate, chain_index),
            )
            chain_x = chain[:, :primary_model.n_modes] @ primary_model.basis.T
            aggregate_chains.append(chain_x @ EQUAL_WEIGHTS)
        aggregate_chains = np.asarray(aggregate_chains)
        rhat_value = split_rhat(aggregate_chains)
        effective_draws = autocorrelation_ess(aggregate_chains)

    calibration_rows.append({
        'replicate': replicate, 'truth_aggregate_log_multiplier': truth_aggregate,
        'laplace_posterior_mean': laplace_aggregate_mean,
        'laplace_posterior_sd': laplace_aggregate_sd,
        'sampling_posterior_mean': float(aggregate_chain0.mean()),
        'sampling_posterior_sd': float(aggregate_chain0.std(ddof=1)),
        'laplace_rank': rank_laplace, 'sampling_rank': rank_sampling,
        'posterior_draws': SBC_DRAWS, 'sampling_cellwise_95_coverage': sampling_cell_coverage,
        'sampling_mean_bracket_evaluations': mean_evaluations,
        'aggregate_split_rhat_4chains': rhat_value,
        'aggregate_effective_draws_4chains': effective_draws,
    })
    if len(calibration_rows) >= 5 or replicate == N_SBC - 1:
        calibration_existing = pd.concat(
            [calibration_existing, pd.DataFrame(calibration_rows)], ignore_index=True
        )
        calibration_existing.to_csv(calibration_path, index=False)
        calibration_rows.clear()
        print(f'  SBC checkpoint through world {replicate + 1}/{N_SBC}', flush=True)

calibration = pd.read_csv(calibration_path).sort_values('replicate').reset_index(drop=True)
if len(calibration) != N_SBC:
    raise RuntimeError(f'SBC rows {len(calibration)} != requested worlds {N_SBC}')


def rank_histogram(frame, column, bins=10):
    indices = np.minimum(
        (frame[column].to_numpy() * bins / (frame.posterior_draws.to_numpy() + 1.0)).astype(int),
        bins - 1,
    )
    counts = np.bincount(indices, minlength=bins)
    statistic, pvalue = chisquare(counts)
    return counts, float(statistic), float(pvalue)


laplace_hist, laplace_chi2, laplace_p = rank_histogram(calibration, 'laplace_rank')
sampling_hist, sampling_chi2, sampling_p = rank_histogram(calibration, 'sampling_rank')
rhat_values = calibration.aggregate_split_rhat_4chains.dropna()
ess_values = calibration.aggregate_effective_draws_4chains.dropna()
sd_ratio = calibration.sampling_posterior_sd / calibration.laplace_posterior_sd

calibration_summary = {
    'strict_prior_predictive_worlds': int(N_SBC),
    'posterior_draws_per_world': int(SBC_DRAWS),
    'laplace_rank_histogram_counts': laplace_hist.tolist(),
    'laplace_rank_chisquare_pvalue': laplace_p,
    'sampling_rank_histogram_counts': sampling_hist.tolist(),
    'sampling_rank_chisquare_pvalue': sampling_p,
    'sampling_cellwise_95_coverage_mean': float(calibration.sampling_cellwise_95_coverage.mean()),
    'sampling_to_laplace_aggregate_sd_ratio_median': float(sd_ratio.median()),
    'sampling_to_laplace_aggregate_sd_ratio_05_95': [float(sd_ratio.quantile(0.05)), float(sd_ratio.quantile(0.95))],
    'four_chain_worlds': int(len(rhat_values)),
    'aggregate_split_rhat_max': float(rhat_values.max()) if len(rhat_values) else None,
    'aggregate_effective_draws_min': float(ess_values.min()) if len(ess_values) else None,
    'calibration_runtime_minutes': float((time.time() - calibration_started) / 60.0),
    'interpretation': (
        'Rank uniformity here evaluates the model-matched computational pipeline only. '
        'Misspecified-world coverage is reported separately and is not called SBC.'
    ),
}
(OUTPUT_DIR / 'strict_prior_predictive_sbc_summary.json').write_text(
    json.dumps(calibration_summary, indent=2)
)
calibration.to_csv(OUTPUT_DIR / 'strict_prior_predictive_sbc.csv', index=False)

print('STRICT CALIBRATION COMPLETE')
print('  Sampling rank p-value:', round(sampling_p, 4))
print('  Laplace rank p-value:', round(laplace_p, 4))
print('  Sampling 95% cell coverage:', round(calibration_summary['sampling_cellwise_95_coverage_mean'], 4))
print('  Maximum 4-chain split R-hat:', calibration_summary['aggregate_split_rhat_max'])


## 7. Claim gates and reviewer-facing status


In [ ]:
def comparison(condition, baseline):
    return paired_comparisons.loc[
        (paired_comparisons.condition == condition) & (paired_comparisons.baseline == baseline)
    ].iloc[0]


primary_vs_fixed = comparison('primary_unresolved_gt1mm', 'best_fixed_equal')
primary_vs_random = comparison('primary_unresolved_gt1mm', 'random')
primary_adaptive = final_summary.loc[
    (final_summary.condition == 'primary_unresolved_gt1mm')
    & (final_summary.policy == 'unweighted_aopt')
].iloc[0]

size_conditions = [spec['condition'] for spec in CONDITION_SPECS if spec['analysis_role'] == 'size_sensitivity']
size_fixed_rows = paired_comparisons.loc[
    paired_comparisons.condition.isin(size_conditions)
    & (paired_comparisons.baseline == 'best_fixed_equal')
]

gate_rows = [
    {
        'gate_group': 'primary', 'gate': 'adaptive_beats_equal_best_fixed_uncertainty',
        'passed': bool(primary_vs_fixed.uncertainty_improvement_ci95_low > 0),
        'observed': float(primary_vs_fixed.uncertainty_improvement_ci95_low),
        'criterion': 'paired bootstrap 95% CI lower bound > 0%',
    },
    {
        'gate_group': 'confirmatory', 'gate': 'adaptive_beats_random_uncertainty',
        'passed': bool(primary_vs_random.uncertainty_improvement_ci95_low > 0),
        'observed': float(primary_vs_random.uncertainty_improvement_ci95_low),
        'criterion': 'paired bootstrap 95% CI lower bound > 0%',
    },
    {
        'gate_group': 'confirmatory', 'gate': 'adaptive_rmse_noninferior_to_random',
        'passed': bool(
            primary_vs_random.rmse_difference_ci95_high
            < 0.05 * primary_vs_random.baseline_rmse_mean
        ),
        'observed': float(primary_vs_random.rmse_difference_ci95_high),
        'criterion': 'upper 95% CI for RMSE harm < 5% of random mean RMSE',
    },
    {
        'gate_group': 'calibration', 'gate': 'primary_laplace_cellwise_95_coverage',
        'passed': bool(0.90 <= primary_adaptive.cellwise_95_coverage_mean <= 0.98),
        'observed': float(primary_adaptive.cellwise_95_coverage_mean),
        'criterion': '0.90 <= mean empirical coverage <= 0.98',
    },
    {
        'gate_group': 'calibration', 'gate': 'sampling_sbc_rank_uniformity',
        'passed': bool(N_SBC >= 50 and sampling_p > 0.01),
        'observed': float(sampling_p),
        'criterion': 'at least 50 model-matched worlds and chi-square p > 0.01',
    },
    {
        'gate_group': 'calibration', 'gate': 'sampling_chain_convergence',
        'passed': bool(len(rhat_values) >= min(N_RHAT_WORLDS, N_SBC) and rhat_values.max() < 1.05),
        'observed': float(rhat_values.max()) if len(rhat_values) else np.nan,
        'criterion': 'maximum aggregate split R-hat across cross-check worlds < 1.05',
    },
    {
        'gate_group': 'calibration', 'gate': 'sampling_effective_draws',
        'passed': bool(len(ess_values) >= min(N_RHAT_WORLDS, N_SBC) and ess_values.min() >= 200),
        'observed': float(ess_values.min()) if len(ess_values) else np.nan,
        'criterion': 'minimum aggregate effective draws across four-chain worlds >= 200',
    },
    {
        'gate_group': 'calibration', 'gate': 'sampling_cellwise_95_coverage',
        'passed': bool(
            0.90 <= calibration_summary['sampling_cellwise_95_coverage_mean'] <= 0.98
        ),
        'observed': float(calibration_summary['sampling_cellwise_95_coverage_mean']),
        'criterion': '0.90 <= mean sampled-posterior coverage <= 0.98',
    },
    {
        'gate_group': 'calibration', 'gate': 'sampling_laplace_aggregate_sd_agreement',
        'passed': bool(0.80 <= float(sd_ratio.median()) <= 1.25),
        'observed': float(sd_ratio.median()),
        'criterion': '0.80 <= median sampling/Laplace aggregate SD ratio <= 1.25',
    },
    {
        'gate_group': 'robustness', 'gate': 'all_size_thresholds_beat_best_fixed',
        'passed': bool((size_fixed_rows.uncertainty_improvement_ci95_low > 0).all()),
        'observed': float(size_fixed_rows.uncertainty_improvement_ci95_low.min()),
        'criterion': 'minimum lower 95% CI across four alternative thresholds > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'half_hidden_signal_beats_best_fixed',
        'passed': bool(comparison('hidden_fraction_0p50', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('hidden_fraction_0p50', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'three_quarter_hidden_signal_beats_best_fixed',
        'passed': bool(comparison('hidden_fraction_0p75', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('hidden_fraction_0p75', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'half_broad_response_beats_best_fixed',
        'passed': bool(comparison('broad_response_0p50', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('broad_response_0p50', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'quarter_broad_response_beats_best_fixed',
        'passed': bool(comparison('broad_response_0p25', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('broad_response_0p25', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'five_mode_model_beats_best_fixed',
        'passed': bool(comparison('latent_basis_kl5', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('latent_basis_kl5', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'restricted_action_domain_beats_best_fixed',
        'passed': bool(comparison('action_domain_70_85', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('action_domain_70_85', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'robustness', 'gate': 'localized_anomaly_beats_best_fixed',
        'passed': bool(comparison('localized_population_anomaly', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('localized_population_anomaly', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%; otherwise report this stress case as unresolved',
    },
    {
        'gate_group': 'robustness', 'gate': 'neighbor_mixing_beats_best_fixed',
        'passed': bool(comparison('neighbor_shell_mixing', 'best_fixed_equal').uncertainty_improvement_ci95_low > 0),
        'observed': float(comparison('neighbor_shell_mixing', 'best_fixed_equal').uncertainty_improvement_ci95_low),
        'criterion': 'lower 95% CI > 0%',
    },
    {
        'gate_group': 'implementation', 'gate': 'all_batch_optimizers_converged',
        'passed': bool(final_metrics.optimizer_success.eq(True).all()),
        'observed': float(final_metrics.optimizer_success.eq(True).mean()),
        'criterion': '100% optimizer success',
    },
]
claim_gates = pd.DataFrame(gate_rows)
claim_gates.to_csv(OUTPUT_DIR / 'reviewer_closure_claim_gates.csv', index=False)

claim_status = {
    'run_complete': True,
    'primary_claim_ready': bool(claim_gates.loc[claim_gates.gate_group == 'primary', 'passed'].all()),
    'calibration_ready': bool(claim_gates.loc[claim_gates.gate_group == 'calibration', 'passed'].all()),
    'all_internal_robustness_gates_pass': bool(claim_gates.loc[claim_gates.gate_group == 'robustness', 'passed'].all()),
    'external_svalbard_validation_complete': False,
    'specific_service_weighting_complete': False,
    'remaining_nonfixable_scope_limits': [
        'single historical Tromso pointing and frozen 57 km slant-range window',
        'no measured antenna side-lobe gain pattern in the current forward model',
        'ORDEM size cases are not a direct diameter-to-detection calibration',
        'other elevations remain prospective rather than historically commanded observations',
    ],
}
(OUTPUT_DIR / 'reviewer_closure_claim_status.json').write_text(json.dumps(claim_status, indent=2))
display(claim_gates)


## 8. Manuscript figures


In [ ]:
plt.style.use('seaborn-v0_8-whitegrid')
fig, axes = plt.subplots(2, 2, figsize=(13.2, 9.2), constrained_layout=True)

ax = axes[0, 0]
for size_key, spec in SIZE_CASES.items():
    totals = H_BY_SIZE[size_key].sum(axis=1)
    ax.plot(ELEVATIONS, totals / totals[CENTER_ACTION], linewidth=2, label=spec['label'])
ax.axvline(75.0, color='black', linestyle='--', linewidth=1)
ax.set(title='A. ORDEM response changes with size assumption', xlabel='Elevation (deg)',
       ylabel='Total response relative to 75 deg')
ax.legend(frameon=False, ncol=2, fontsize=9)

ax = axes[0, 1]
primary_steps = step_metrics.loc[step_metrics.condition == 'primary_unresolved_gt1mm']
for policy in POLICIES:
    group = primary_steps.loc[primary_steps.policy == policy].groupby('step').equal_uncertainty_fraction
    mean = group.mean()
    se = group.std(ddof=1) / np.sqrt(group.count())
    ax.plot(mean.index, mean, color=POLICY_COLORS[policy], linewidth=2, label=POLICY_LABELS[policy])
    ax.fill_between(mean.index, mean - 1.96 * se, mean + 1.96 * se,
                    color=POLICY_COLORS[policy], alpha=0.12)
ax.set(title='B. Frozen primary learning curve', xlabel='One-hour observations',
       ylabel='Equal-weighted uncertainty remaining')
ax.legend(frameon=False, fontsize=9)

ax = axes[1, 0]
forest = paired_comparisons.loc[paired_comparisons.baseline == 'best_fixed_equal'].copy()
forest['order'] = forest.condition.map({spec['condition']: i for i, spec in enumerate(CONDITION_SPECS)})
forest = forest.sort_values('order', ascending=False)
y = np.arange(len(forest))
means = forest.uncertainty_improvement_percent.to_numpy()
low = forest.uncertainty_improvement_ci95_low.to_numpy()
high = forest.uncertainty_improvement_ci95_high.to_numpy()
colors = ['#E45756' if role == 'primary' else '#4C78A8' for role in forest.analysis_role]
for index in range(len(forest)):
    ax.errorbar(
        means[index], y[index], xerr=[[means[index] - low[index]], [high[index] - means[index]]],
        fmt='o', color=colors[index], ecolor=colors[index], elinewidth=1.7, capsize=3, markersize=6,
    )
ax.axvline(0, color='black', linewidth=1)
ax.set_yticks(y, forest.display_label)
ax.set(title='C. Adaptive reduction versus target-matched best fixed',
       xlabel='Uncertainty reduction after 24 hours (%)')
ax.grid(axis='y', visible=False)

ax = axes[1, 1]
edges = np.arange(11)
ax.step(edges[:-1] + 0.5, laplace_hist, where='mid', linewidth=2,
        color='#7F7F7F', label=f'Laplace (p={laplace_p:.3f})')
ax.step(edges[:-1] + 0.5, sampling_hist, where='mid', linewidth=2,
        color='#E45756', label=f'Posterior samples (p={sampling_p:.3f})')
ax.axhline(N_SBC / 10.0, color='black', linestyle='--', linewidth=1)
ax.set(title='D. Strict model-matched aggregate SBC', xlabel='Rank bin', ylabel='Independent worlds')
ax.set_xticks(np.arange(10) + 0.5, np.arange(1, 11))
ax.legend(frameon=False, fontsize=9)

fig.suptitle('Reviewer-closure: empirical detector bridge and internal robustness', fontsize=15, fontweight='bold')
fig.savefig(OUTPUT_DIR / 'figure_reviewer_closure_main_2x2.png', dpi=220, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_reviewer_closure_main_2x2.pdf', bbox_inches='tight')
plt.show()
fig, axes = plt.subplots(2, 2, figsize=(12.8, 8.8), constrained_layout=True)

ax = axes[0, 0]
event_labels = ['Strict catalog', 'Supported', 'Unresolved']
event_values = [46, 3, 106]
bars = ax.bar(event_labels, event_values, color=['#4C78A8', '#72B7B2', '#E45756'])
ax.bar_label(bars)
ax.set(title='A. Frozen radar-event decomposition', ylabel='Consolidated events')

ax = axes[0, 1]
rcs = catalog_recovery.loc[catalog_recovery.RCS_SIZE != 'ALL'].copy()
rcs['sort_order'] = rcs.RCS_SIZE.map({'SMALL': 0, 'MEDIUM': 1, 'LARGE': 2})
rcs = rcs.sort_values('sort_order')
values = rcs.strict_recovery_fraction.to_numpy()
ax.errorbar(
    np.arange(len(rcs)), values,
    yerr=[values - rcs.wilson_ci95_low.to_numpy(), rcs.wilson_ci95_high.to_numpy() - values],
    fmt='o', color='#4C78A8', capsize=4, markersize=7,
)
ax.set_xticks(np.arange(len(rcs)), rcs.RCS_SIZE)
ax.set_ylim(bottom=0)
ax.set(title='B. Catalog-first recovery depends on RCS class', xlabel='Catalog RCS class',
       ylabel='Strict passage recovery fraction')

ax = axes[1, 0]
for size_key, spec in SIZE_CASES.items():
    group = identifiability.loc[identifiability.size_case == size_key]
    ax.plot(group.component, group.cumulative_energy_fraction, marker='o', label=spec['label'])
ax.axhline(0.99, color='black', linestyle='--', linewidth=1)
ax.axvline(5, color='black', linestyle=':', linewidth=1)
ax.set(xlim=(1, 10), ylim=(0, 1.02), title='C. Response-library identifiability',
       xlabel='Singular component', ylabel='Cumulative response energy')
ax.legend(frameon=False, fontsize=8, ncol=2)

ax = axes[1, 1]
ax.scatter(calibration.laplace_posterior_sd, calibration.sampling_posterior_sd,
           color='#E45756', alpha=0.75, s=28)
limit = max(calibration.laplace_posterior_sd.max(), calibration.sampling_posterior_sd.max()) * 1.05
ax.plot([0, limit], [0, limit], color='black', linestyle='--', linewidth=1)
ax.set(xlim=(0, limit), ylim=(0, limit), title='D. Sampling check of Laplace aggregate uncertainty',
       xlabel='Laplace posterior SD', ylabel='Sampled posterior SD')

fig.suptitle('Detector evidence, identifiability, and inference diagnostics', fontsize=15, fontweight='bold')
fig.savefig(OUTPUT_DIR / 'figure_reviewer_closure_diagnostics_2x2.png', dpi=220, bbox_inches='tight')
fig.savefig(OUTPUT_DIR / 'figure_reviewer_closure_diagnostics_2x2.pdf', bbox_inches='tight')
plt.show()


## 9. Final manifest and plain-language status


In [ ]:
readme = f'# research reviewer-closure run `{config_hash}`\n\nThis run freezes one primary methodological result before sensitivity analysis:\nequal-weighted adaptive A-optimal scheduling versus the best equal-weighted fixed elevation after 24 one-hour observations.\n\nWhat changed from the earlier asset-weighted run:\n\n- The primary signal count scale uses 106 unresolved events, not all 155 events.\n- The 46 strict and 3 supported catalog associations are excluded from the hidden-population signal.\n- The detector null-derived upper false-event rate remains an explicit background.\n- Five completed ORDEM size thresholds are tested; none is declared the exact EISCAT size cutoff.\n- Possible unresolved contamination and broad/side-lobe response are explicit sensitivity cases.\n- A five-mode latent model tests the effect of limited response rank.\n- Strict SBC uses only model-matched prior-predictive worlds.\n- Elliptical-slice posterior samples cross-check the Fisher-Laplace approximation.\n- Satellite/service weights do not enter the primary result.\n\nThis run cannot create missing empirical information. Svalbard validation, a measured antenna side-lobe model,\nand any service-specific satellite weighting remain separate work. The empirical scope is one Tromso day,\none historical pointing, and the frozen 57 km slant-range window.\n\nThe primary adaptive improvement versus best fixed is\n{primary_vs_fixed.uncertainty_improvement_percent:.2f}%\n(95% paired bootstrap CI {primary_vs_fixed.uncertainty_improvement_ci95_low:.2f}% to\n{primary_vs_fixed.uncertainty_improvement_ci95_high:.2f}%).\n\nUse `reviewer_closure_claim_gates.csv` and `reviewer_closure_claim_status.json` to determine which statements\nare supported. `_SUCCESS.json` means the notebook finished, not that every scientific gate passed.\n'
(OUTPUT_DIR / 'README_reviewer_closure.md').write_text(readme)
output_files = [path for path in OUTPUT_DIR.iterdir() if path.is_file() and path.name not in {'output_manifest.json', '_SUCCESS.json'}]
manifest_rows = []
for path in sorted(output_files):
    manifest_rows.append({'file': path.name, 'bytes': path.stat().st_size, 'sha256': sha256_file(path)})
manifest = {'config_hash': config_hash, 'created_at_utc': pd.Timestamp.now(tz='UTC').isoformat(), 'input_hashes': input_hashes, 'outputs': manifest_rows}
(OUTPUT_DIR / 'output_manifest.json').write_text(json.dumps(manifest, indent=2))
success = {'success': True, 'config_hash': config_hash, 'output_directory': str(OUTPUT_DIR), 'primary_claim_ready': claim_status['primary_claim_ready'], 'calibration_ready': claim_status['calibration_ready'], 'all_internal_robustness_gates_pass': claim_status['all_internal_robustness_gates_pass'], 'files_in_manifest': len(manifest_rows)}
(OUTPUT_DIR / '_SUCCESS.json').write_text(json.dumps(success, indent=2))
print('\nSUCCESS — reviewer-closure run is complete.')
print(json.dumps(success, indent=2))
print('\nPrimary adaptive improvement versus best fixed:')
print(f'{primary_vs_fixed.uncertainty_improvement_percent:.2f}% (95% CI {primary_vs_fixed.uncertainty_improvement_ci95_low:.2f}% to {primary_vs_fixed.uncertainty_improvement_ci95_high:.2f}%)')
